# Profile: forward + backward, B=16, GA=32

Two pipelines: prefix LM and phylo encoder-only. For prefix LM, also break the
forward into its custom functions and report % of forward time per function.

Method: `torch.cuda.synchronize()` + `time.time()`. Discard first iteration
(warmup). Average the rest.


## Setup

In [1]:
import os, time, torch
DEVICE = torch.device(f"cuda:{int(os.environ.get('LOCAL_RANK', 0))}")
import random
from gLM.models import ProteinBertModel
from gLM.models.protein_modernbert_phylo import ProteinModernBertPrefixLM
# from gLM.attention_mask.prefixlm_flash2 import (
#     prefixlm_forward_flash,
#     run_encoder_flash,
#     build_pack_indices,
#     build_rope_cos_sin_both,
#     build_layer_type_list,
#     build_unpack_fn,
#     _global_two_call,
#     _local_two_call,
# )


from gLM.attention_mask.prefixlm_flash import (
    prefixlm_forward_flash,
    
)
from gLM.tokenizers import PhyloTokenizerLoader
from gLM.collator import PhyloCollator
from gLM.collator.prefixlm_collator import PrefixLMCollator
from gLM.dataset import Uniref90ArrowEvalDatasetForLMDB, Uniref90ArrowDatasetForLMDB

TRAIN_PATH = "/gpfs/data/brandeslab/Data/uniref/uniref90_clusters_arrow/train"
LMDB_PATH  = "/gpfs/data/brandeslab/Data/uniref/uniref100_merged.lmdb"
TOKENIZER  = "./phylo_char_tokenizer_with_bos"

tokenizer = PhyloTokenizerLoader(TOKENIZER)

B, GA, n_steps = 16, 32, 10


/gpfs/data/brandeslab/User/as12267/.conda/envs/huggingface_bert_cu126/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Build prefix LM model + batch

In [2]:
ds_prefix = Uniref90ArrowDatasetForLMDB(
    dataset_path=TRAIN_PATH, training_type="phylo_encoder_decoder", lmdb_path=LMDB_PATH)
collator_prefix = PrefixLMCollator(tokenizer=tokenizer, max_seq_len=2048, has_pid=False)

INDICES = random.sample(range(len(ds_prefix)), B)
batch_prefix = collator_prefix([ds_prefix[i] for i in INDICES])

# for i in INDICES[:5]:
#     s1, s2 = ds_prefix[i]
#     print(i, len(s1), len(s2), len(s1) + len(s2))


batch_prefix = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch_prefix.items()}
print(f"Prefix LM batch shape: {batch_prefix['input_ids'].shape}")
print(f"  keys: {list(batch_prefix.keys())}")
print(f"  per-sample seq lengths (non-pad): "
      f"{(batch_prefix['input_ids'] != tokenizer.pad_token_id).sum(dim=1).tolist()}")
print(f"  prefix_lengths: {batch_prefix['prefix_lengths'].tolist()}")
print(f"  suffix_lengths: {(batch_prefix['input_ids'] != tokenizer.pad_token_id).sum(dim=1).sub(batch_prefix['prefix_lengths']).tolist()}")
print(f"  labels mask (count of non-(-100)): "
      f"{(batch_prefix['labels'] != -100).sum(dim=1).tolist()}")
print(f"  total tokens (sum non-pad): {(batch_prefix['input_ids'] != tokenizer.pad_token_id).sum().item()}")
print(f"  padded tokens: {(batch_prefix['input_ids'] == tokenizer.pad_token_id).sum().item()}")
print(f"  padding fraction: {(batch_prefix['input_ids'] == tokenizer.pad_token_id).float().mean().item():.2%}")

model_prefix = ProteinModernBertPrefixLM(vocab_size=tokenizer.vocab_size, tokenizer=tokenizer).build()
model_prefix.gradient_checkpointing_enable()
model_prefix.to(DEVICE)
model_prefix.train()
opt_prefix = torch.optim.AdamW(model_prefix.parameters(), lr=3e-4)


Prefix LM batch shape: torch.Size([16, 2048])
  keys: ['input_ids', 'labels', 'prefix_lengths']


You are attempting to use Flash Attention 2 without specifying a torch dtype. This might lead to unexpected behaviour


  per-sample seq lengths (non-pad): [1035, 535, 787, 376, 2048, 781, 1301, 348, 1181, 899, 393, 823, 1129, 285, 747, 897]
  prefix_lengths: [506, 268, 394, 188, 1022, 395, 658, 175, 591, 450, 197, 412, 565, 143, 374, 449]
  suffix_lengths: [529, 267, 393, 188, 1026, 386, 643, 173, 590, 449, 196, 411, 564, 142, 373, 448]
  labels mask (count of non-(-100)): [528, 266, 392, 187, 1025, 385, 642, 172, 589, 448, 195, 410, 563, 141, 372, 447]
  total tokens (sum non-pad): 13565
  padded tokens: 19203
  padding fraction: 58.60%


## Build phylo encoder-only model + batch

In [3]:
ds_phylo = Uniref90ArrowDatasetForLMDB(
    dataset_path=TRAIN_PATH, training_type="phylo_encoder_only", lmdb_path=LMDB_PATH)
collator_phylo = PhyloCollator(tokenizer=tokenizer, training_type="phylo_encoder_only", max_seq_len=2048)

batch_phylo = collator_phylo([ds_phylo[i] for i in INDICES])
batch_phylo = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch_phylo.items()}
print(f"Phylo batch shape: {batch_phylo['input_ids'].shape}")
print(f"Per-sample non-pad lengths: {(batch_phylo['input_ids'] != tokenizer.pad_token_id).sum(dim=1).tolist()}")
print(f"  keys: {list(batch_phylo.keys())}")
print(f"  per-sample seq lengths (non-pad): "
      f"{(batch_phylo['input_ids'] != tokenizer.pad_token_id).sum(dim=1).tolist()}")
print(f"  attention_mask sums: {batch_phylo['attention_mask'].sum(dim=1).tolist()}")
print(f"  labels mask (count of non-(-100)): "
      f"{(batch_phylo['labels'] != -100).sum(dim=1).tolist()}")
print(f"  percent_identity: {batch_phylo['percent_identity'].tolist()}")
print(f"  total tokens (sum non-pad): {(batch_phylo['input_ids'] != tokenizer.pad_token_id).sum().item()}")
print(f"  padded tokens: {(batch_phylo['input_ids'] == tokenizer.pad_token_id).sum().item()}")
print(f"  padding fraction: {(batch_phylo['input_ids'] == tokenizer.pad_token_id).float().mean().item():.2%}")

model_phylo = ProteinBertModel(
    vocab_size=tokenizer.vocab_size, tokenizer=tokenizer, attn_implementation="flash_attention_2").build()
model_phylo.gradient_checkpointing_enable()
model_phylo.to(DEVICE)
model_phylo.train()
opt_phylo = torch.optim.AdamW(model_phylo.parameters(), lr=3e-4)

print(f"Phylo batch shape: {batch_phylo['input_ids'].shape}")


Phylo batch shape: torch.Size([16, 1050])
Per-sample non-pad lengths: [528, 266, 392, 187, 1050, 393, 649, 173, 589, 448, 195, 410, 563, 141, 372, 447]
  keys: ['input_ids', 'attention_mask', 'labels', 'percent_identity']
  per-sample seq lengths (non-pad): [528, 266, 392, 187, 1050, 393, 649, 173, 589, 448, 195, 410, 563, 141, 372, 447]
  attention_mask sums: [528, 266, 392, 187, 1050, 393, 649, 173, 589, 448, 195, 410, 563, 141, 372, 447]
  labels mask (count of non-(-100)): [528, 266, 392, 187, 1050, 393, 649, 173, 589, 448, 195, 410, 563, 141, 372, 447]
  percent_identity: [89.96212005615234, 99.24812316894531, 93.62245178222656, 93.04812622070312, 99.23809814453125, 90.58524322509766, 88.906005859375, 92.48554992675781, 97.62309265136719, 99.10713958740234, 97.43589782714844, 92.19512176513672, 99.64476013183594, 82.97872161865234, 95.96774291992188, 97.98657989501953]
  total tokens (sum non-pad): 6803
  padded tokens: 9997
  padding fraction: 59.51%
Using flash_attention_2 atten

## Forward + backward, both pipelines

One full optimizer step = `GA` forward+backward passes + 1 optimizer step.
We run 3 such steps and discard the first.


In [4]:
def time_step(forward_fn, opt, batch, GA):
    """Time forward + backward for GA micro-batches + 1 optimizer step.
        1 optimizer step = 32 forward + 32 backward passes
    Returns (fwd_total_ms, bwd_total_ms, opt_ms)."""
    fwd_total = bwd_total = 0.0

    # repeat the forward and backward pass GA times
    for _ in range(GA):
        # for each micro-batch, time the forward pass 
        torch.cuda.synchronize(); t0 = time.time()
        loss = forward_fn(batch)
        torch.cuda.synchronize(); t1 = time.time()
        # for each micro-batch, time the backward pass
        (loss / GA).backward()
        torch.cuda.synchronize(); t2 = time.time()
        # add up forward and backward times across GA micro-batches
        fwd_total += (t1 - t0) * 1000
        bwd_total += (t2 - t1) * 1000
    # after GA micro-batches, time the optimizer step
    torch.cuda.synchronize(); t0 = time.time()
    opt.step(); opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    opt_ms = (time.time() - t0) * 1000

    return fwd_total, bwd_total, opt_ms # return total forward and backward time across GA micro-batches, and opt step time

In [5]:
# Forward function for modernbert with alignment 
# runs the model in bf16 mixed precision and returns the loss
def fwd_phylo(b, model):
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        return model(**b).loss

# Forward function for modernbert prefix LM 
# calls custom forward function 
# runs the model in bf16 mixed precision and returns the loss
def fwd_prefix(b, model):
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        loss, _ = prefixlm_forward_flash(model, b, DEVICE)
    return loss


def run_pipeline(name, fwd_fn, model, opt, batch, GA, n_steps=n_steps):
    """Run n_steps full optimizer steps, time each, drop the warmup step."""
    results = []
    for i in range(n_steps):
        # Wrap fwd_fn in a lambda so time_step can call it with one arg
        fwd, bwd, op = time_step(lambda b: fwd_fn(b, model), opt, batch, GA)
        results.append((fwd, bwd, op))
        marker = " (warmup, dropped)" if i == 0 else ""
        print(f"  [{name}] step {i+1}{marker}: fwd={fwd:.0f}ms bwd={bwd:.0f}ms opt={op:.0f}ms")
    # Drop first (warmup), average rest
    fwd, bwd, op = [sum(x) / (n_steps-1) for x in zip(*results[1:])]
    print(f"  [{name}] avg (n={n_steps-1}): fwd={fwd:.0f}ms bwd={bwd:.0f}ms opt={op:.0f}ms total={fwd+bwd+op:.0f}ms")
    return fwd, bwd, op


print("=== Prefix LM ===")
prefix_fwd, prefix_bwd, prefix_opt = run_pipeline(
    "prefix", fwd_prefix, model_prefix, opt_prefix, batch_prefix, GA)

print("\n=== Phylo encoder-only ===")
phylo_fwd, phylo_bwd, phylo_opt = run_pipeline(
    "phylo", fwd_phylo, model_phylo, opt_phylo, batch_phylo, GA)

print("\n=== Comparison (per optimizer step, GA=32) ===")
prefix_total = prefix_fwd + prefix_bwd + prefix_opt
phylo_total  = phylo_fwd  + phylo_bwd  + phylo_opt
print(f"  prefix LM (T={batch_prefix['input_ids'].shape[1]}): "
      f"{prefix_total:.0f}ms total ({prefix_fwd:.0f} fwd + {prefix_bwd:.0f} bwd + {prefix_opt:.0f} opt)")
print(f"  phylo     (T={batch_phylo['input_ids'].shape[1]}): "
      f"{phylo_total:.0f}ms total ({phylo_fwd:.0f} fwd + {phylo_bwd:.0f} bwd + {phylo_opt:.0f} opt)")

if prefix_total > phylo_total:
    print(f"  prefix LM is {prefix_total/phylo_total:.2f}x slower than phylo")
else:
    print(f"  phylo is {phylo_total/prefix_total:.2f}x slower than prefix LM")

=== Prefix LM ===
  [prefix] step 1 (warmup, dropped): fwd=4192ms bwd=5653ms opt=115ms
  [prefix] step 2: fwd=2846ms bwd=5407ms opt=7ms
  [prefix] step 3: fwd=2862ms bwd=5432ms opt=7ms
  [prefix] step 4: fwd=2876ms bwd=5451ms opt=7ms
  [prefix] step 5: fwd=2888ms bwd=5468ms opt=7ms
  [prefix] step 6: fwd=2898ms bwd=5490ms opt=7ms
  [prefix] step 7: fwd=2910ms bwd=5498ms opt=7ms
  [prefix] step 8: fwd=2909ms bwd=5508ms opt=7ms
  [prefix] step 9: fwd=2912ms bwd=5515ms opt=7ms
  [prefix] step 10: fwd=2914ms bwd=5520ms opt=7ms
  [prefix] avg (n=9): fwd=2891ms bwd=5476ms opt=7ms total=8374ms

=== Phylo encoder-only ===
  [phylo] step 1 (warmup, dropped): fwd=29256ms bwd=3821ms opt=8ms
  [phylo] step 2: fwd=469ms bwd=1405ms opt=7ms
  [phylo] step 3: fwd=469ms bwd=1405ms opt=7ms
  [phylo] step 4: fwd=473ms bwd=1408ms opt=7ms
  [phylo] step 5: fwd=468ms bwd=1425ms opt=7ms
  [phylo] step 6: fwd=468ms bwd=1411ms opt=7ms
  [phylo] step 7: fwd=469ms bwd=1413ms opt=7ms
  [phylo] step 8: fwd=469ms b

## Compare PrefixLM and Alignment ModernBERT on similar number of unpadded tokens

In [6]:
import random
import time
import torch

B = 16
TARGET_TOTAL_TOKENS = 6000   # total non-pad tokens you want each batch to have

def collect_with_lengths(ds, collator, n_probe=2000, seed=42):
    """Probe n_probe samples; for each, run collator on a single sample and record its non-pad length."""
    # sample 2000 random indices 
    rng = random.Random(seed)
    indices = rng.sample(range(len(ds)), n_probe)
    out = []
    # run collator on each sample, record its non-pad length
    for idx in indices:
        try:
            single = collator([ds[idx]])
        except Exception:
            continue
        L = (single['input_ids'] != tokenizer.pad_token_id).sum().item()
        out.append((idx, L))
    return out


def pick_batch_for_token_budget(samples, B, target_total, tol=0.00):
    """Greedily pick B samples whose non-pad lengths sum to ~target_total.
    Sort by length, then pick samples spaced through the distribution
    so the sum lands near target_total."""
    # Aim for average length = target_total / B
    target_avg = target_total / B
    # Sort by absolute distance from target_avg
    ranked = sorted(samples, key=lambda x: abs(x[1] - target_avg))
    # pick B number of samples that are closest to target_avg
    chosen = ranked[:B]
    total = sum(L for _, L in chosen)
    if abs(total - target_total) / target_total > tol:
        print(f"  warning: picked total={total}, target={target_total} "
              f"(off by {abs(total-target_total)/target_total:.1%})")
    return [idx for idx, _ in chosen], total

In [7]:
# probe each pipeline once
print("Probing prefix...")
prefix_probe = collect_with_lengths(ds_prefix, collator_prefix, n_probe=2000)
print(f"  got {len(prefix_probe)} samples, length range "
      f"{min(l for _,l in prefix_probe)}–{max(l for _,l in prefix_probe)}")

print("Probing phylo...")
phylo_probe = collect_with_lengths(ds_phylo, collator_phylo, n_probe=2000)
print(f"  got {len(phylo_probe)} samples, length range "
      f"{min(l for _,l in phylo_probe)}–{max(l for _,l in phylo_probe)}")

Probing prefix...
  got 2000 samples, length range 79–2048
Probing phylo...
  got 2000 samples, length range 38–2048


In [8]:
# pick batches
prefix_indices, prefix_total = pick_batch_for_token_budget(prefix_probe, B, TARGET_TOTAL_TOKENS)
phylo_indices,  phylo_total  = pick_batch_for_token_budget(phylo_probe,  B, TARGET_TOTAL_TOKENS)

batch_prefix = collator_prefix([ds_prefix[i] for i in prefix_indices])
batch_phylo  = collator_phylo([ds_phylo[i] for i in phylo_indices])

batch_prefix = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch_prefix.items()}
batch_phylo  = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch_phylo.items()}

In [9]:
# verify
p_nonpad = (batch_prefix['input_ids'] != tokenizer.pad_token_id).sum().item()
ph_nonpad = (batch_phylo['input_ids']  != tokenizer.pad_token_id).sum().item()

print(f"\n=== Prefix batch ===")
print(f"  shape: {tuple(batch_prefix['input_ids'].shape)}")
print(f"  non-pad tokens (total): {p_nonpad}")
print(f"  padding fraction: {(batch_prefix['input_ids']==tokenizer.pad_token_id).float().mean():.1%}")
print(f"  per-sample non-pad lengths: "
      f"{(batch_prefix['input_ids'] != tokenizer.pad_token_id).sum(dim=1).tolist()}")
print(f"  prefix_lengths (seq2 portion): {batch_prefix['prefix_lengths'].tolist()}")
suffix_lens = ((batch_prefix['input_ids'] != tokenizer.pad_token_id).sum(dim=1)
               - batch_prefix['prefix_lengths']).tolist()
print(f"  suffix_lengths (seq1 portion): {suffix_lens}")
print(f"  raw len(s1), len(s2) per sample:")
for idx in prefix_indices:
    s1, s2 = ds_prefix[idx]
    print(f"    idx={idx}: len(s1)={len(s1)}, len(s2)={len(s2)}, sum={len(s1)+len(s2)}")

print(f"\n=== Phylo batch ===")
print(f"  shape: {tuple(batch_phylo['input_ids'].shape)}")
print(f"  non-pad tokens (total): {ph_nonpad}")
print(f"  padding fraction: {(batch_phylo['input_ids']==tokenizer.pad_token_id).float().mean():.1%}")
print(f"  per-sample non-pad lengths: "
      f"{(batch_phylo['input_ids'] != tokenizer.pad_token_id).sum(dim=1).tolist()}")
print(f"  raw len(a1), len(a2) per sample:")
for idx in phylo_indices:
    a1, a2 = ds_phylo[idx][0], ds_phylo[idx][1]
    print(f"    idx={idx}: len(a1)={len(a1)}, len(a2)={len(a2)}")

print(f"\nToken ratio (prefix/phylo): {p_nonpad/ph_nonpad:.2f}x")


=== Prefix batch ===
  shape: (16, 389)
  non-pad tokens (total): 6008
  padding fraction: 3.5%
  per-sample non-pad lengths: [375, 375, 375, 375, 376, 377, 373, 373, 389, 373, 373, 377, 377, 378, 371, 371]
  prefix_lengths (seq2 portion): [188, 188, 188, 188, 188, 189, 187, 187, 195, 187, 187, 189, 183, 188, 186, 186]
  suffix_lengths (seq1 portion): [187, 187, 187, 187, 188, 188, 186, 186, 194, 186, 186, 188, 194, 190, 185, 185]
  raw len(s1), len(s2) per sample:
    idx=40515368: len(s1)=186, len(s2)=186, sum=372
    idx=40483595: len(s1)=186, len(s2)=186, sum=372
    idx=40514536: len(s1)=186, len(s2)=186, sum=372
    idx=38090558: len(s1)=186, len(s2)=186, sum=372
    idx=40400793: len(s1)=187, len(s2)=186, sum=373
    idx=39897505: len(s1)=187, len(s2)=187, sum=374
    idx=40589442: len(s1)=185, len(s2)=185, sum=370
    idx=40660770: len(s1)=185, len(s2)=185, sum=370
    idx=39735014: len(s1)=193, len(s2)=193, sum=386
    idx=40629504: len(s1)=185, len(s2)=185, sum=370
    idx=3

In [10]:
print("=== Prefix LM ===")
prefix_fwd, prefix_bwd, prefix_opt = run_pipeline(
    "prefix", fwd_prefix, model_prefix, opt_prefix, batch_prefix, GA)

print("\n=== Phylo encoder-only ===")
phylo_fwd, phylo_bwd, phylo_opt = run_pipeline(
    "phylo", fwd_phylo, model_phylo, opt_phylo, batch_phylo, GA)

# comparison (same as before)
prefix_total_ms = prefix_fwd + prefix_bwd + prefix_opt
phylo_total_ms  = phylo_fwd  + phylo_bwd  + phylo_opt
print(f"\nprefix (T={batch_prefix['input_ids'].shape[1]}, "
      f"non-pad={p_nonpad}): {prefix_total_ms:.0f}ms")
print(f"phylo  (T={batch_phylo['input_ids'].shape[1]}, "
      f"non-pad={ph_nonpad}): {phylo_total_ms:.0f}ms")

if prefix_total_ms > phylo_total_ms:
    print(f"  prefix is {prefix_total_ms/phylo_total_ms:.2f}x slower than phylo")
else:
    print(f"  phylo is {phylo_total_ms/prefix_total_ms:.2f}x slower than prefix")

=== Prefix LM ===
  [prefix] step 1 (warmup, dropped): fwd=710ms bwd=1370ms opt=7ms
  [prefix] step 2: fwd=693ms bwd=1371ms opt=7ms
  [prefix] step 3: fwd=693ms bwd=1371ms opt=7ms
  [prefix] step 4: fwd=694ms bwd=1374ms opt=7ms
  [prefix] step 5: fwd=695ms bwd=1375ms opt=7ms
  [prefix] step 6: fwd=695ms bwd=1377ms opt=7ms
  [prefix] step 7: fwd=696ms bwd=1377ms opt=7ms
  [prefix] step 8: fwd=697ms bwd=1380ms opt=7ms
  [prefix] step 9: fwd=696ms bwd=1379ms opt=7ms
  [prefix] step 10: fwd=698ms bwd=1381ms opt=7ms
  [prefix] avg (n=9): fwd=695ms bwd=1376ms opt=7ms total=2078ms

=== Phylo encoder-only ===
  [phylo] step 1 (warmup, dropped): fwd=418ms bwd=1244ms opt=7ms
  [phylo] step 2: fwd=417ms bwd=1244ms opt=7ms
  [phylo] step 3: fwd=419ms bwd=1246ms opt=7ms
  [phylo] step 4: fwd=418ms bwd=1247ms opt=7ms
  [phylo] step 5: fwd=418ms bwd=1248ms opt=7ms
  [phylo] step 6: fwd=419ms bwd=1249ms opt=7ms
  [phylo] step 7: fwd=419ms bwd=1250ms opt=7ms
  [phylo] step 8: fwd=419ms bwd=1250ms opt=7

## Prefix LM forward breakdown

Inline copy of `prefixlm_forward_flash` + `run_encoder_flash` with timing
around each function. Run on **one** micro-batch, 10 iterations, drop iter 0.

The bf16 autocast wrapper matches what `prefixlm_forward_flash` does
internally — without it, the matmuls run in fp32 and timings don't match
the real forward.


In [13]:
from torch.nn import CrossEntropyLoss
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb
from gLM.attention_mask.prefixlm_flash import (
    build_scatter_plan,
    build_rope_both,
    build_layer_type_list,
    scatter_unpack,
    _flash_two_call_global,
    _flash_two_call_local,
)

def timed_forward(model, batch, device, timings):
    """One forward pass. Appends per-function ms to `timings` dict."""
    base = model.module if hasattr(model, "module") else model
    encoder = base.model
    config = encoder.config

    def tic():
        torch.cuda.synchronize()
        return time.time()
    def toc(name, t0):
        torch.cuda.synchronize()
        timings.setdefault(name, []).append((time.time() - t0) * 1000)

    input_ids = batch["input_ids"].to(device)
    labels    = batch["labels"].to(device)
    prefix_lengths = batch["prefix_lengths"].to(device)
    B, T = input_ids.shape
    num_heads = config.num_attention_heads
    head_dim  = config.hidden_size // num_heads

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        # ── Setup ─────────────────────────────────────────────────────
        t = tic()
        padding_mask = (input_ids != config.pad_token_id)
        seq_lens    = padding_mask.sum(dim=1).int().to(device)
        prefix_lens = prefix_lengths.int().to(device)
        toc("setup_seqlens", t)

        t = tic()
        plan            = build_scatter_plan(B, T, seq_lens, prefix_lens, device)
        prefix_flat_idx = plan["prefix_flat_idx"]
        suffix_flat_idx = plan["suffix_flat_idx"]
        full_flat_idx   = plan["full_flat_idx"]
        cu_prefix       = plan["cu_prefix"]
        cu_suffix_q     = plan["cu_suffix_q"]
        cu_full         = plan["cu_full"]
        max_prefix      = plan["max_prefix"]
        max_suffix      = plan["max_suffix"]
        max_full        = plan["max_full"]
        has_suffix      = int(plan["suffix_lens"].max()) > 0
        toc("build_scatter_plan", t)

        t = tic()
        layer_is_global = build_layer_type_list(config)
        local_window    = getattr(config, "local_attention", 512)
        toc("build_layer_type_list", t)

        t = tic()
        hidden_states = encoder.embeddings(input_ids=input_ids)
        toc("embeddings", t)

        t = tic()
        theta_global = config.global_rope_theta
        theta_local  = getattr(config, "local_rope_theta", 10000.0)
        (cos_global, sin_global), (cos_local, sin_local) = build_rope_both(
            T, head_dim, theta_global, theta_local, device, hidden_states.dtype)
        toc("build_rope_both", t)

        position_ids = torch.arange(T, device=device).unsqueeze(0).expand(B, -1)
        dropout_p    = config.attention_dropout if model.training else 0.0

        # ── Layer loop ────────────────────────────────────────────────
        for layer_idx, layer in enumerate(encoder.layers):
            is_global = layer_is_global[layer_idx]

            t = tic()
            attn   = layer.attn
            normed = layer.attn_norm(hidden_states)
            qkv = attn.Wqkv(normed).view(B, T, 3, num_heads, head_dim)
            q, k, v = qkv.unbind(dim=2)
            toc("layer_qkv_proj", t)

            t = tic()
            cos = cos_global if is_global else cos_local
            sin = sin_global if is_global else sin_local
            q_t, k_t = apply_rotary_pos_emb(
                q.transpose(1, 2), k.transpose(1, 2), cos, sin,
                position_ids=position_ids, unsqueeze_dim=1)
            q_f = q_t.transpose(1, 2).reshape(B*T, num_heads, head_dim).to(torch.bfloat16)
            k_f = k_t.transpose(1, 2).reshape(B*T, num_heads, head_dim).to(torch.bfloat16)
            v_f = v.reshape(B*T, num_heads, head_dim).to(torch.bfloat16)
            toc("layer_rope_apply", t)

            t = tic()
            q_pre = q_f[prefix_flat_idx]; k_pre = k_f[prefix_flat_idx]; v_pre = v_f[prefix_flat_idx]
            k_ful = k_f[full_flat_idx];   v_ful = v_f[full_flat_idx]
            q_suf = q_f[suffix_flat_idx] if has_suffix else None
            toc("layer_pack", t)

            t = tic()
            if is_global:
                out_prefix, out_suffix = _flash_two_call_global(
                    q_pre, k_pre, v_pre, q_suf, k_ful, v_ful,
                    cu_prefix, cu_suffix_q, cu_full,
                    max_prefix, max_suffix, max_full, dropout_p)
                toc("layer_global_attn", t)
            else:
                out_prefix, out_suffix = _flash_two_call_local(
                    q_pre, k_pre, v_pre, q_suf, k_ful, v_ful,
                    cu_prefix, cu_suffix_q, cu_full,
                    max_prefix, max_suffix, max_full,
                    local_window, dropout_p)
                toc("layer_local_attn", t)

            t = tic()
            attn_out = scatter_unpack(
                out_prefix, out_suffix if has_suffix else None,
                prefix_flat_idx, suffix_flat_idx,
                B, T, num_heads, head_dim,
                dtype=hidden_states.dtype, device=device)
            toc("layer_unpack", t)

            t = tic()
            attn_out = attn.out_drop(attn.Wo(attn_out.reshape(B, T, -1)))
            hidden_states = hidden_states + attn_out
            toc("layer_out_proj", t)

            t = tic()
            hidden_states = hidden_states + layer.mlp(layer.mlp_norm(hidden_states))
            toc("layer_mlp", t)

        # ── Final ─────────────────────────────────────────────────────
        t = tic()
        hidden_states = encoder.final_norm(hidden_states)
        toc("final_norm", t)

        t = tic()
        logits = base.decoder(base.head(hidden_states))
        toc("head_and_decoder", t)

        t = tic()
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        loss = CrossEntropyLoss(ignore_index=-100)(
            shift_logits.view(-1, base.config.vocab_size), shift_labels.view(-1))
        toc("loss", t)

    return loss


# Run 11 iterations, drop first
timings = {}
for i in range(11):
    iter_timings = {}
    loss = timed_forward(model_prefix, batch_prefix, DEVICE, iter_timings)
    if i > 0:  # skip warmup
        for k, v in iter_timings.items():
            timings.setdefault(k, []).extend(v)
    model_prefix.zero_grad(set_to_none=True)

# Aggregate
import statistics
n_iters = 10
print(f"\n{'Component':<28} {'Total (ms)':>12} {'Per iter':>12} {'% of fwd':>10}")
print("-" * 65)

# Sum each component's total time across all 10 iters
totals = {k: sum(v) for k, v in timings.items()}
grand = sum(totals.values())

# Sort by total descending
for k, total in sorted(totals.items(), key=lambda x: -x[1]):
    per_iter = total / n_iters
    pct = 100 * total / grand
    print(f"{k:<28} {total:>12.1f} {per_iter:>12.2f} {pct:>9.1f}%")

print("-" * 65)
print(f"{'TOTAL':<28} {grand:>12.1f} {grand/n_iters:>12.2f}    100.0%")


Component                      Total (ms)     Per iter   % of fwd
-----------------------------------------------------------------
layer_mlp                            93.2         9.32      36.0%
layer_rope_apply                     50.0         5.00      19.3%
layer_qkv_proj                       23.5         2.35       9.1%
layer_local_attn                     19.2         1.92       7.4%
layer_out_proj                       19.2         1.92       7.4%
layer_unpack                         13.7         1.37       5.3%
layer_pack                           11.1         1.11       4.3%
build_scatter_plan                   10.0         1.00       3.8%
layer_global_attn                     9.5         0.95       3.7%
build_rope_both                       2.7         0.27       1.1%
head_and_decoder                      2.3         0.23       0.9%
loss                                  1.6         0.16       0.6%
embeddings                            1.1         0.11       0.4%
setup_seq

## How to read this

- **Per iter** is mean ms per forward pass for that component.
- Per-layer rows (`layer_*`) are summed across all 12 layers per forward,
  so divide by 12 if you want per-layer time. Global (4 layers) and local
  (8 layers) attention are reported separately.
- Setup rows (`setup_*`, `build_*`, `embeddings`, `build_rope_*`) run once
  per forward.
- TOTAL should match the per-step forward time from the first profile
  divided by GA, within ~10% (sync overhead from extra timing calls).
